In [1]:
import nibabel as nib        # For nifti files
import numpy as np           # For matrix math
import SimpleITK as sitk     # For N4 correction
import torchio as tio        # For deep learning
from dcm2niix import main as dcm2niix_run
from pathlib import Path
from dotenv import load_dotenv
import os
import ants
from nilearn.datasets import MNI152_FILE_PATH
import torch

#pipeline:
# 1. motion correction and n4 bias field correction

# 2. Skull stripping

# 3. spatial normalization

# 4. Intensity normalization

# 5. resizing (use placeholder values to signify dims)

# 6. Gaussian filters to smooth and denois data

# preserve as a 5d tensor for volumetric architecture



c:\Users\brand\Downloads\work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
import subprocess
import tempfile
import torch
import torchio as tio
import ants


def preprocess_mri_volume(
    nifti_path: str,
    mni_template_path: str,
    target_shape: tuple = (128, 128, 128),
    use_hdbet: bool = True
) -> torch.Tensor:
    """
    3D MRI Preprocessing pipeline using ANTsPy for registration
    and TorchIO/HD-BET for normalization and formatting.
    
    Returns:
        torch.Tensor: 5D Volumetric Tensor of shape (1, 1, D, H, W)
    """
    subject = tio.Subject(t1w=tio.ScalarImage(nifti_path))
    subject = tio.ToCanonical()(subject)
    
    # Save a clean temp copy for ANTs reading
    with tempfile.TemporaryDirectory() as tmp_dir:
        input_temp = os.path.join(tmp_dir, "input_ras.nii.gz")
        subject.t1w.save(input_temp)
        
        moving_ants = ants.image_read(input_temp)
        template_ants = ants.image_read(mni_template_path)

        n4_corrected_ants = ants.n4_bias_field_correction(moving_ants)

        reg = ants.registration(
            fixed=template_ants,
            moving=n4_corrected_ants,
            type_of_transform='Affine'
        )
        registered_ants = reg['warpedmovout']

        reg_temp = os.path.join(tmp_dir, "registered_mni.nii.gz")
        ants.image_write(registered_ants, reg_temp)
        subject.t1w = tio.ScalarImage(reg_temp)

    if use_hdbet:
        with tempfile.TemporaryDirectory() as tmp_dir:
            in_file = os.path.join(tmp_dir, "input.nii.gz")
            out_file = os.path.join(tmp_dir, "output.nii.gz")
            mask_file = os.path.join(tmp_dir, "output_mask.nii.gz")
            
            subject.t1w.save(in_file)
            subprocess.run(["hd-bet", "-i", in_file, "-o", out_file], check=True)
            
            brain_mask = tio.LabelMap(mask_file)
            subject.add_image(brain_mask, 'brain_mask')
            subject.t1w.data = subject.t1w.data * subject.brain_mask.data
    else:
        # Simple foreground threshold fallback
        foreground_mask = tio.ForegroundMask(mask_name='brain_mask')
        subject = foreground_mask(subject)
        subject.t1w.data = subject.t1w.data * subject.brain_mask.data

    target_resample = tio.Resize(target_shape)
    subject = target_resample(subject)

    # Re-extract foreground mask on the final grid
    subject = tio.ForegroundMask(mask_name='final_brain_mask')(subject)
    z_norm = tio.ZNormalization(masking_method='final_brain_mask')
    subject = z_norm(subject)

    tensor_4d = subject.t1w.data        # Shape: (1, D, H, W)
    tensor_5d = tensor_4d.unsqueeze(0)  # Shape: (1, 1, D, H, W)

    return tensor_5d


# Example Usage
if __name__ == "__main__":
    NIFTI_INPUT = "path/to/subject_T1w.nii.gz"
    MNI_TEMPLATE = "path/to/mni152_1mm.nii.gz"
    DIMS = (128, 128, 128)

    output_tensor = preprocess_mri_volume(
        nifti_path=NIFTI_INPUT,
        mni_template_path=MNI_TEMPLATE,
        target_shape=DIMS,
        use_hdbet=True
    )

    print(f"Final Tensor Shape: {output_tensor.shape}")  # (1, 1, 128, 128, 128)

In [ ]:
from pathlib import Path
import torch

DIMS = (128,128,128)

patient_directory = Path(r'string')
output_directory = Path(r'path/to/processed_tensors')
output_directory.mkdir(parents=True, exist_ok=True)

for nifti_file in patient_directory.rglob('*.nii.gz'):
    if nifti_file.name.startswith('.'):
        continue

    processed_tensor = preprocess_mri_volume(
        nifti_path=str(nifti_file),
        mni_template_path='samplefilepath',
        target_shape=DIMS,
        use_hdbet=True
    )

    # Save the resulting 5D tensor for model training
    clean_stem = nifti_file.name.replace('.nii.gz', '')
    save_path = output_directory / f"{clean_stem}_preprocessed.pt"
    torch.save(processed_tensor, save_path)

In [ ]:
# 1. Path to your top ADNI folder

load_dotenv()
folder_path = os.getenv('ADNI_FOLDER_PATH')
raw_root = Path(folder_path)

# 2. Path to where you want all converted .nii.gz files saved
output_root = Path("data/nifti_raw")

# Loop through every Participant folder inside cn cohort/ADNI
for subject_dir in raw_root.iterdir():
    if subject_dir.is_dir():
        print(f"Converting subject: {subject_dir.name}")
        
        # Create a matching subject directory in your output folder
        subj_output = output_root / subject_dir.name
        subj_output.mkdir(parents=True, exist_ok=True)
        
        # dcm2niix will automatically crawl down into MPRAGE -> Visits -> Cryptic ID -> DICOMs
        dcm2niix_run([
            "-z", "y",                 # Compress output to .nii.gz
            "-f", "%i_%p_%s",          # Filename style: ParticipantID_Protocol_Series
            "-o", str(subj_output),    # Destination folder for this participant
            str(subject_dir)           # Input directory (this participant's root folder)
        ])

print("All participant conversions complete!")